In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 89.6 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kagglehub

In [ ]:
!kaggle datasets download -d sdaiancai/sada2022 -f "train.csv" -p "/content/drive/MyDrive/Arabic_Dialect_Classification/"

In [ ]:
!unzip /content/drive/MyDrive/Arabic_Dialect_Classification/train.csv.zip -d /content/drive/MyDrive/Arabic_Dialect_Classification/

In [ ]:
import numpy as np # linear algebra
#np.random.seed(40)
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px # data visualization
import re # preprocessing text
from nltk.corpus import stopwords # get stops words
from sklearn.metrics import confusion_matrix  # Compute confusion matrix to evaluate the accuracy of a classification.
from plotly.subplots import make_subplots  # plot sub plots
import plotly.graph_objects as go # data visualization
import matplotlib.pyplot as plt # data visualization

from tensorflow.keras.utils import to_categorical # Converts a class vector (integers) to binary class matrix.
from sklearn.preprocessing import LabelEncoder # Encode target labels with value between 0 and n_classes-1.
from collections import Counter # collection where elements are stored as dictionary keys and their counts are stored as dictionary values
import tensorflow as tf # deep learning library
from tensorflow.keras.preprocessing.text import Tokenizer # to convert text into tokens
from tensorflow.keras.preprocessing.sequence import pad_sequences  # to add padding into tokens sequances

#########
from sklearn.model_selection import train_test_split # to split data into train and test
# machine learning algorithms
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import RocCurveDisplay

#########
import warnings
warnings.filterwarnings('ignore')

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
color = ['#3c8283']

In [ ]:
# load dataset
df = pd.read_csv('/content/drive/MyDrive/Arabic_Dialect_Classification/train.csv')

In [ ]:
df.shape

(241834, 16)

In [ ]:
# show the head of the dataset
df.head(3)

,Unnamed: 0,FileName,ShowName,FullFileLength,SegmentID,SegmentLength,SegmentStart,SegmentEnd,SpeakerAge,SpeakerGender,SpeakerDialect,Speaker,Environment,GroundTruthText,ProcessedText,Category
0,0,batch_1/6k_SBA_100_0.wav,الجفاف يقتل الندى,605.3,6k_SBA_100_0-seg_249_720-251_840,2.12,249.72,251.84,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ووضّح كلامك يا مغيث,ووضح كلامك يا مغيث,درامي
1,1,batch_1/6k_SBA_100_0.wav,الجفاف يقتل الندى,605.3,6k_SBA_100_0-seg_252_700-255_300,2.60,252.70,255.30,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ترى رأسي ما عاد يتحمّل ألغازك.,ترى راسي ما عاد يتحمل الغازك,درامي
2,2,batch_1/6k_SBA_100_0.wav,الجفاف يقتل الندى,605.3,6k_SBA_100_0-seg_256_010-258_180,2.17,256.01,258.18,Elderly -- كبير في السن,Male,Najdi,Speaker2متحدث,Clean -- نظيف,سلامة رأسك يا أبو مسامح.,سلامة راسك يا ابو مسامح,درامي


In [ ]:
# drop unused columns
df.drop(['Unnamed: 0','FileName','FullFileLength','SegmentStart','SegmentEnd'], axis= 1, inplace=True)
df.head(3)

,ShowName,SegmentID,SegmentLength,SpeakerAge,SpeakerGender,SpeakerDialect,Speaker,Environment,GroundTruthText,ProcessedText,Category
0,الجفاف يقتل الندى,6k_SBA_100_0-seg_249_720-251_840,2.12,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ووضّح كلامك يا مغيث,ووضح كلامك يا مغيث,درامي
1,الجفاف يقتل الندى,6k_SBA_100_0-seg_252_700-255_300,2.60,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ترى رأسي ما عاد يتحمّل ألغازك.,ترى راسي ما عاد يتحمل الغازك,درامي
2,الجفاف يقتل الندى,6k_SBA_100_0-seg_256_010-258_180,2.17,Elderly -- كبير في السن,Male,Najdi,Speaker2متحدث,Clean -- نظيف,سلامة رأسك يا أبو مسامح.,سلامة راسك يا ابو مسامح,درامي


In [ ]:
df.isna().sum()

,0
ShowName,0
SegmentID,0
SegmentLength,0
SpeakerAge,0
SpeakerGender,0
SpeakerDialect,0
Speaker,0
Environment,0
GroundTruthText,1
ProcessedText,207


In [ ]:
print("Cound of duplicated rows in the dataset: ",df.duplicated().sum())

Cound of duplicated rows in the dataset:  0


In [ ]:
# drop rows when Ground Truth Text is null
df.drop(df[df.GroundTruthText.isna()].index, inplace=True)

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df.head(3)

,ShowName,SegmentID,SegmentLength,SpeakerAge,SpeakerGender,SpeakerDialect,Speaker,Environment,GroundTruthText,ProcessedText,Category
0,الجفاف يقتل الندى,6k_SBA_100_0-seg_249_720-251_840,2.12,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ووضّح كلامك يا مغيث,ووضح كلامك يا مغيث,درامي
1,الجفاف يقتل الندى,6k_SBA_100_0-seg_252_700-255_300,2.60,Elderly -- كبير في السن,Male,Najdi,Speaker1متحدث,Clean -- نظيف,ترى رأسي ما عاد يتحمّل ألغازك.,ترى راسي ما عاد يتحمل الغازك,درامي
2,الجفاف يقتل الندى,6k_SBA_100_0-seg_256_010-258_180,2.17,Elderly -- كبير في السن,Male,Najdi,Speaker2متحدث,Clean -- نظيف,سلامة رأسك يا أبو مسامح.,سلامة راسك يا ابو مسامح,درامي


In [ ]:
df.describe(include='object').T

,count,unique,top,freq
ShowName,241833,64,حكايات بابا فرحان,37577
SegmentID,241833,241833,6k_v_SBA_999_1-seg_519_760-520_640,1
SpeakerAge,241833,6,Adult -- بالغ,153702
SpeakerGender,241833,4,Male,116715
SpeakerDialect,241833,14,Najdi,90657
Speaker,241833,13,Speaker1متحدث,70598
Environment,241833,4,Music -- موسيقى,88979
GroundTruthText,241833,225995,هاهاها,214
ProcessedText,241627,219982,هاهاها,434
Category,241833,11,كوميدي,69172


In [ ]:
df.describe(include='number').T

,count,mean,std,min,25%,50%,75%,max
SegmentLength,241833.0,6.216942,7.41109,0.5,1.47,3.214999,7.52,161.31


SegmentLength represents the length of a segment (likely in seconds).

Most segments are short:

50% are under ~3.2 seconds

75% are under ~7.5 seconds

The maximum is very large (161.31), meaning there are some very long outliers.

The mean (6.22) is higher than the median (3.21) → suggests a right-skewed distribution due to long segments.

This could affect modeling or padding if using audio or text sequences of different lengths.

In [ ]:
# Counts of samples for each Dialect
speaker_dialect_counts = df['SpeakerDialect'].value_counts()
fig = px.bar(x = speaker_dialect_counts.index,
             y = speaker_dialect_counts.values,
             title ="Counts of samples for each Dialect ",
             labels = {'x':'Speaker Dialect', 'y':'Counts of samples in the dataset'},
             color_discrete_sequence = color * len(speaker_dialect_counts),
             template = "simple_white",
             text=['{}'.format(p) for p in speaker_dialect_counts.values]
            )
fig.show()

Custom Preprocessing Implementation sourced from social media—we implemented two custom Python classes: PreprocessTweets and TweetsTokenizing. These classes provide a structured and layered approach to cleaning, normalizing, and preparing the text before feeding it into classification models.

In [ ]:
class PreprocessTweets:
    def __init__ (self, text):
        self.text = text

    def normalize_letters(self):
        self.text = re.sub("[إأآا]", "ا", self.text)
        self.text = re.sub("ى", "ي", self.text)
        self.text = re.sub("ؤ", "ء", self.text)
        self.text = re.sub("ئ", "ء", self.text)
        self.text = re.sub("ة", "ه", self.text)
        self.text = re.sub("گ", "ك", self.text)
        self.text =  re.sub(r'(.)\1+', r'\1', self.text)

    def remove_tashkeel(self):
        tashkeel = re.compile("""
                                  ّ    | # Tashdid
                                  َ    | # Fatha
                                  ً    | # Tanwin Fath
                                  ُ    | # Damma
                                  ٌ    | # Tanwin Damm
                                  ِ    | # Kasra
                                  ٍ    | # Tanwin Kasr
                                  ْ    | # Sukun
                                ـ     # Tatwil/Kashida
                                    """, re.VERBOSE)
        self.text = re.sub(tashkeel, '', self.text)

    def sub_chars(self):
        chlist = [chr(i) for i in range(1569, 1611)] + [chr(i) for i in range(1646, 1749)]
        self.text = ''.join([i if i in chlist else ' ' for i in self.text])
        self.text = ' '.join([u'{}'.format(i) for i in self.text.split() if len(i) > 1])
        self.text = re.sub(r'\d+', ' ', self.text)

    def remove_symbols_spaces_and_english(self):
        symbols1 = re.compile('[/(){}\[\]\|,;]')
        symbols2 = re.compile("[@|(|)|\||:|>|<|_|#|\.|+|÷|×|'|!|\?|٪|؟\|&|;|\*|[|]|{|}|-|،|_|’|;|!|:|^|&|%|/]")
        arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''

        emojies = re.compile("["
                            u"\U0001F600-\U0001F64F"
                            u"\U0001F300-\U0001F5FF"
                            u"\U0001F680-\U0001F6FF"
                            u"\U0001F1E0-\U0001F1FF"
                            u"\U00002500-\U00002BEF"
                            u"\U00002702-\U000027B0"
                            u"\U00002702-\U000027B0"
                            u"\U000024C2-\U0001F251"
                            u"\U0001f926-\U0001f937"
                            u"\U00010000-\U0010ffff"
                            u"\u2640-\u2642"
                            u"\u0640"
                            u"\u2600-\u2B55"
                            u"\u200d"
                            u"\u23cf"
                            u"\u23e9"
                            u"\u231a"
                            u"\ufe0f"  # dingbats
                            u"\u3030"
                            "]+", re.UNICODE)

        self.text = re.sub(emojies, ' ', self.text)
        self.text =  re.sub(symbols1, ' ', self.text)
        self.text =  re.sub(symbols2, ' ', self.text)

        translator = str.maketrans('', '', arabic_punctuations)
        self.text = self.text.translate(translator)

        self.text = self.text.replace('"', " ")
        self.text = self.text.replace('…', " ")
        self.text =  re.sub(r'\s*[A-Za-z]+\b', ' ' , self.text)

        while '  ' in self.text:
            self.text = self.text.replace('  ', ' ')

    def preprocessing_pipeline(self):
        self.normalize_letters()
        self.remove_tashkeel()
        self.sub_chars()
        self.remove_symbols_spaces_and_english()

        return self.text

In [ ]:
class TweetsTokenizing:

    def __init__(self, text):
        self.text = text
        self.tokens = []

    def tokenize_text(self):
        tokens = word_tokenize(self.text)
        self.tokens = [token.strip('') for token in tokens]

    def removeStopWords(self):
        stopwords_list = stopwords.words('arabic')
        listStopWords = stopwords_list
        self.tokens = [i for i in self.tokens if not i in listStopWords]

    def remove_repeated_characters(self):
        repeat_pattern = re.compile(r'(\w*)(\w)\2(\w*)')
        match_substitution = r'\1\2\3'

        def replace(old_word):
            if wordnet.synsets(old_word):
                return old_word
            new_word = repeat_pattern.sub(match_substitution, old_word)
            return replace(new_word) if new_word != old_word else new_word

        self.tokens = [replace(word) for word in self.tokens]

    def tokenize_pipeline(self):
        self.tokenize_text()
        self.removeStopWords()
        self.remove_repeated_characters()

        return self.tokens

In [ ]:
df["Cleanedtext"] = df["GroundTruthText"].apply(lambda text : PreprocessTweets(str(text)).preprocessing_pipeline())

This code applies a preprocessing function to every entry in the GroundTruthText column and stores the cleaned version in a new column Cleanedtext.

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.tokenize import RegexpTokenizer
import numpy as np
from sklearn import preprocessing

nltk.download('all')
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.tokenize import RegexpTokenizer

NLTK stands for Natural Language Toolkit — it's a powerful open-source Python library for working with human language data (text). It’s one of the oldest and most widely used libraries for Natural Language Processing (NLP) tasks.

In [ ]:
df["Cleaned_text"] = df["Cleanedtext"].apply(lambda text : TweetsTokenizing(text).tokenize_pipeline())

This code takes already cleaned text and tokenizes it, storing the result in a new DataFrame column.

So, now your DataFrame (df) has:

GroundTruthText: original text

Cleanedtext: cleaned version of the original

Cleaned_text: tokenized version of the cleaned text

In [ ]:
df[['GroundTruthText','Cleaned_text']].head(20)

,GroundTruthText,Cleaned_text
0,ووضّح كلامك يا مغيث,"[وضح, كلامك, مغيث]"
1,ترى رأسي ما عاد يتحمّل ألغازك.,"[تري, راسي, يتحمل, الغازك]"
2,سلامة رأسك يا أبو مسامح.,"[سلامه, راسك, ابو, مسامح]"
3,ما يصير يا أبو مسامح تخلّي البنت في البيت دون ...,"[يصير, ابو, مسامح, تخلي, البنت, البيت, امها]"
4,ويش فيها لقعدت في البيت دون أمها ما هو ده بيت ...,"[ويش, لقعدت, البيت, امها, ده, بيت, ابوها, بيت,..."
5,إلّا بيت أبوها لكن لا تنسى إنّك حطّمت حياتها.,"[الا, بيت, ابوها, تنسي, انك, حطمت, حياتها]"
6,يبغى إنسى أنت منين تجيبلي هالكلام الفاضي.,"[يبغي, انسي, انت, منين, تجيبلي, هالكلام, الفاضي]"
7,ما هو فاضي.,[فاضي]
8,طردت أمها و طردت أخوها زيد.,"[طردت, امها, طردت, اخوها, زيد]"
9,وإذا قعدت في البيت في خطرٍ عليك ولا يصير تخلّي...,"[واذا, قعدت, البيت, خطر, يصير, تخليها, غرفه, م..."


In [ ]:
words = [word for tokens in df["Cleaned_text"] for word in tokens]
sentence_lens = [len(tokens) for tokens in df["Cleaned_text"]]

VOC = sorted(list(set(words)))

print("%s words total, with a vocabulary size of %s" % (len(words), len(VOC)))
print("Max sentence length is %s" % max(sentence_lens))

2592922 words total, with a vocabulary size of 155619
Max sentence length is 99


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
counter = Counter(words)
counter.most_common(20)

[('انا', 48266),
 ('اله', 45394),
 ('الي', 35743),
 ('علي', 31914),
 ('انت', 23035),
 ('يعني', 22826),
 ('ان', 17334),
 ('ايش', 16095),
 ('شاء', 12457),
 ('يلا', 12353),
 ('طيب', 12130),
 ('واله', 11320),
 ('اي', 10769),
 ('انه', 10279),
 ('ايه', 10006),
 ('ابو', 9073),
 ('وش', 8939),
 ('هاهاها', 8819),
 ('عشان', 8424),
 ('الحين', 8186)]

used the Counter class from Pythons collections module to find the most common words

In [ ]:
import pandas as pd

# Assume 'df' is your preprocessed DataFrame containing 'text' and 'label' columns
# If you're working with Hugging Face Dataset, you can convert it to a pandas DataFrame
df_cleaned = pd.DataFrame({
    'Cleaned_text': df['Cleanedtext'],  # This is the cleaned text column
    'SpeakerDialect': df['SpeakerDialect']  # This is the label column
})

In [ ]:
# Save the cleaned dataset to a specific location in your Google Drive
file_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/cleaned_dataset.csv'  # Path to save the file in your Google Drive

# Save the cleaned DataFrame as a CSV file
df_cleaned.to_csv(file_path, encoding="utf-8-sig", index=False)
# Print confirmation
print(f"Data saved to {file_path}")

Data saved to /content/drive/MyDrive/Arabic_Dialect_Classification/cleaned_dataset.csv


# Installing the necessary packages

In [ ]:
# Install the necessary packages for Arabic NLP processing
!pip install arabert  # Arabic BERT Preprocessing
!pip install transformers  # Hugging Face's Transformer Library for NLP
!pip install farasapy  # Farasa Library for Arabic NLP tasks
!pip install pyarabic  # Python library for Arabic text processing

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Arabic_Dialect_Classification/cleaned_dataset.csv", engine="python")

In [ ]:
data

,Cleaned_text,SpeakerDialect
0,وضح كلامك يا مغيث,Najdi
1,تري راسي ما عاد يتحمل الغازك,Najdi
2,سلامه راسك يا ابو مسامح,Najdi
3,ما يصير يا ابو مسامح تخلي البنت في البيت دون ...,Najdi
4,ويش فيها لقعدت في البيت دون امها ما هو ده بيت...,Najdi
...,...,...
241828,لازم تشيلها عشان تصير سعيد في حياتك,Najdi
241829,انا سويت كل هذا علي شانك لا واله مريم مريم عط...,More than 1 speaker اكثر من متحدث
241830,مريم,Najdi
241831,لا لا,Najdi


In [ ]:
data["SpeakerDialect"].unique()


array(['Najdi', 'More than 1 speaker اكثر من متحدث', 'Unknown', 'Khaliji',
       'Hijazi', 'ModernStandardArabic', 'Notapplicable', 'Egyptian',
       'Levantine', 'Yemeni', 'Maghrebi', 'Janubi', 'Shamali', 'Iraqi'],
      dtype=object)

In [ ]:
# Mapping dialects to numbers
map_label = {
    'Najdi': 0,
    'More than 1 speaker اكثر من متحدث': 1,
    'Unknown': 2,
    'Khaliji': 3,
    'Hijazi': 4,
    'ModernStandardArabic': 5,
    'Notapplicable': 6,
    'Egyptian': 7,
    'Levantine': 8,
    'Yemeni': 9,
    'Maghrebi': 10,
    'Janubi': 11,
    'Shamali': 12,
    'Iraqi': 13
}

# Mapping numbers back to dialects
label_map = {v: k for k, v in map_label.items()}

#print(map_label)
#print(label_map)

In [ ]:
data

,Cleaned_text,SpeakerDialect
0,وضح كلامك يا مغيث,Najdi
1,تري راسي ما عاد يتحمل الغازك,Najdi
2,سلامه راسك يا ابو مسامح,Najdi
3,ما يصير يا ابو مسامح تخلي البنت في البيت دون ...,Najdi
4,ويش فيها لقعدت في البيت دون امها ما هو ده بيت...,Najdi
...,...,...
241828,لازم تشيلها عشان تصير سعيد في حياتك,Najdi
241829,انا سويت كل هذا علي شانك لا واله مريم مريم عط...,More than 1 speaker اكثر من متحدث
241830,مريم,Najdi
241831,لا لا,Najdi


# Preprocess Arabic Text using AraBERT

In [ ]:
from arabert.preprocess import ArabertPreprocessor  # Import AraBERT Preprocessor

model_name = "bert-base-arabert"  # Choose AraBERT model
arabert_prep = ArabertPreprocessor(model_name=model_name)  # Initialize preprocessor

100%|██████████| 241M/241M [03:14<00:00, 1.24MiB/s]


[2025-05-09 14:55:06,383 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [ ]:
from sklearn.model_selection import train_test_split
# Rename columns if necessary
data = data.rename(columns={'SpeakerDialect': 'dialect', 'Cleaned_text': 'text'})


In [ ]:
data

,text,dialect
0,وضح كلامك يا مغيث,Najdi
1,تري راسي ما عاد يتحمل الغازك,Najdi
2,سلامه راسك يا ابو مسامح,Najdi
3,ما يصير يا ابو مسامح تخلي البنت في البيت دون ...,Najdi
4,ويش فيها لقعدت في البيت دون امها ما هو ده بيت...,Najdi
...,...,...
241828,لازم تشيلها عشان تصير سعيد في حياتك,Najdi
241829,انا سويت كل هذا علي شانك لا واله مريم مريم عط...,More than 1 speaker اكثر من متحدث
241830,مريم,Najdi
241831,لا لا,Najdi


In [ ]:
# Split into train (80%) and test (20%)
dataset, test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['dialect'])

# Display dataset shapes
print("Train set size:", dataset.shape)
print("Test set size:", test_data.shape)

Train set size: (193466, 2)
Test set size: (48367, 2)


Apply AraBERT Preprocessing to Text


In [ ]:
dataset["text"]=dataset["text"].apply(lambda x:arabert_prep.preprocess(x))

In [ ]:
test_data["text"]=test_data["text"].apply(lambda x:arabert_prep.preprocess(x))

In [ ]:
dataset

,text,dialect
70238,حنضبطه بس,Najdi
99201,و+ يشقي و+ يتعب علشان يجمع شي ب+ عيل +ه الي من...,More than 1 speaker اكثر من متحدث
134830,امرك سيدي ال+ منظم خدو و+ دو +ه داخل ال+ سجن ا...,More than 1 speaker اكثر من متحدث
76711,غير واضح هذا ال+,Khaliji
118122,ب+ حط كدا زعتر أو خبز مقلي الي تحب +ون أنا حاط...,Najdi
...,...,...
201450,بابا نعم عرف +ت تحل مسال +ه ال+ رياضي +ات لا ب...,More than 1 speaker اكثر من متحدث
41312,و+ أنا جيت ك+ اخطب بنت ك+ فاطم +ه اه و+ اله +ا...,More than 1 speaker اكثر من متحدث
153933,لا سعيد لا تخاف نحن +ا مع بعض رايح +ين نساعد ب...,More than 1 speaker اكثر من متحدث
221728,كيف عرف +ت,ModernStandardArabic


In [ ]:
# Define the path where you want to save the datasets
train_data_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/dataset.csv'
test_data_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/test_data.csv'

# Save the datasets to Google Drive
dataset.to_csv(train_data_path, index=False)
test_data.to_csv(test_data_path, index=False)

print(f"Datasets saved to:\n{train_data_path}\n{test_data_path}")


# Importing the necessary packages

In [ ]:
from arabert.preprocess import ArabertPreprocessor
from sklearn.metrics import (accuracy_score, f1_score,recall_score)
from torch.utils.data import  Dataset
from transformers import (AutoConfig, AutoModelForSequenceClassification,
                        AutoTokenizer, BertTokenizer, Trainer,
                        TrainingArguments)
from transformers.data.processors.utils import InputFeatures

In [ ]:
#chose bert model
model_name = 'aubmindlab/bert-base-arabert'
#asafaya/bert-base-arabic
#UBC-NLP/ARBERT
#UBC-NLP/MARBERT
#bert-base-multilingual-uncased
num_labels = 14
max_length = 120

## To work using PyTorch we need to create a classification dataset to load the data

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, text, target, model_name, max_len, label_map):
      super(ClassificationDataset).__init__()

      self.text = text
      self.target = target
      self.tokenizer_name = model_name
      self.tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer
      self.max_len = max_len # Set maximum text length
      self.label_map = label_map


    def __len__(self):
      return len(self.text)

    def __getitem__(self,item):
      text = str(self.text[item])
      text = " ".join(text.split())

      inputs = self.tokenizer(
          text,
          max_length=self.max_len,
          padding='max_length',
          truncation=True
        )
      return InputFeatures(**inputs,label= self.target[item])

In [ ]:

dataset['dialect'] = dataset['dialect'].map(map_label)
test_data['dialect'] = test_data['dialect'].map(map_label)


In [ ]:
test_data=test_data[test_data['dialect'].isnull()==False]
dataset=dataset[dataset['dialect'].isnull()==False]

In [ ]:
# Convert labels to integers
dataset['dialect'] = dataset['dialect'].astype(int)
test_data['dialect'] = test_data['dialect'].astype(int)

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 193466 entries, 70238 to 124320
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     193466 non-null  object
 1   dialect  193466 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 4.4+ MB


## Creating datasets
This part of the code initializes PyTorch datasets for training and testing using the ClassificationDataset class.

In [ ]:
train_dataset = ClassificationDataset(
    dataset['text'].to_list(),   # Convert text column to a list (training data)
    dataset['dialect'].to_list(),  # Convert dialect labels to a list (training labels)
    model_name,  # Name of the pretrained model (e.g., 'bert-base-arabert')
    max_length,  # Maximum token length for input sequences
    map_label  # Dictionary mapping dialect names to numerical labels
)
test_dataset = ClassificationDataset(
    test_data['text'].to_list(),   # Convert text column to a list (test data)
    test_data['dialect'].to_list(),  # Convert dialect labels to a list (test labels)
    model_name,  # Name of the pretrained model
    max_length,  # Maximum sequence length
    map_label  # Label mapping dictionary
)


tokenizer_config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/717k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
(len(train_dataset))

193466

## Create a function that return a pretrained model ready to do classification

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, return_dict=True, num_labels=num_labels)

## Metrics

In [ ]:
def compute_metrics(p): #p should be of type EvalPrediction
  preds = np.argmax(p.predictions, axis=1)
  assert len(preds) == len(p.label_ids)
  macro_f1 = f1_score(p.label_ids,preds,average='macro')
  macro_recall = recall_score(p.label_ids,preds,average='macro')
  acc = accuracy_score(p.label_ids,preds)
  return {
      'macro_f1' : macro_f1,
      'accuracy': acc,
      'recall':macro_recall
  }

## Training arguments

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Arabic_Dialect_Classification/train",
    adam_epsilon=1e-8,
    learning_rate=5e-6,
    fp16=True,  # استخدام حساب النقطة العائمة 16 بت إذا كان لديك V100/T4
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=7,  # تقليل عدد العصور لتجنب Overfitting
    warmup_ratio=0.1,  # استخدام Warmup لتحسين الاستقرار
    do_eval=True,
    weight_decay=0.01 ,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=5,  # تقليل عدد النماذج المحفوظة لتوفير المساحة
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=[],
)

## Creating the trainer

In [ ]:
trainer = Trainer(
    model = model_init(),
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Tarining

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Recall
1,0.995800,1.226792,0.233131,0.559803,0.227104
2,0.988000,1.225399,0.237031,0.564434,0.226616
3,0.952100,1.233453,0.238627,0.562925,0.232507
4,0.909200,1.262767,0.242265,0.562615,0.232126
5,0.884200,1.275853,0.243099,0.558191,0.234801
6,0.867500,1.300591,0.246240,0.559431,0.235616
7,0.837900,1.313780,0.248661,0.556909,0.238216


TrainOutput(global_step=21161, training_loss=0.92251360278393, metrics={'train_runtime': 4217.0171, 'train_samples_per_second': 321.142, 'train_steps_per_second': 5.018, 'total_flos': 8.352180357105312e+16, 'train_loss': 0.92251360278393, 'epoch': 7.0})

## Saving the model

In [ ]:
import os
model_save_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/model_checkpoint'
# Set the model configuration for label mappings
trainer.model.config.label2id = map_label
trainer.model.config.id2label = label_map
trainer.save_model(os.path.join(model_save_path, 'checkpoint-1'))
train_dataset.tokenizer.save_pretrained(os.path.join(model_save_path, 'checkpoint-1'))

In [ ]:
# Define the path in Google Drive
model_save_path = '/content/drive/MyDrive/Arabic_Dialect_Classification/model_checkpoint'

# Set the model configuration for label mappings
trainer.model.config.label2id = map_label
trainer.model.config.id2label = label_map

# Save the model to Google Drive
trainer.save_model(model_save_path)

# Save the tokenizer to Google Drive
train_dataset.tokenizer.save_pretrained(model_save_path)
